# Notebook 03 — Taxi Metric Design

## Purpose

This notebook is the first real analytical notebook in the project.

The goal is not to make a causal claim. The goal is to design a credible taxi-side monitoring layer that can support operational review.

At this stage, we focus only on taxi data and evaluate whether a pickup zone × day table can capture useful signals of mobility friction.

## Analytical objectives

This notebook is used to:

- verify that study-window filtering is enforced correctly
- confirm that invalid trips are removed before grouped metrics are computed
- inspect the distribution of key trip variables
- construct robust daily zone-level taxi metrics
- evaluate which metrics are stable enough to carry forward into later analysis

## Important framing decisions

Current project choices:

- raw data remains untouched
- cleaning happens before aggregation
- the unit of analysis is pickup zone × day
- low trip count alone does not define friction
- early friction indicators should rely on robust statistics such as medians and upper-tail metrics rather than means

## Known data-quality risks

We already know that raw TLC taxi files can contain timestamps outside the intended month or study window.

Because of that, the study window must be enforced explicitly using `pickup_datetime` before aggregation.

We also apply trip-level validity filters before computing zone-day metrics.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from nyc_mobility_friction.paths import get_project_paths

paths = get_project_paths()

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)

## Load taxi data

For this notebook, we should avoid loading raw multi-month trip data unless needed.

The preferred input is an already cleaned taxi table or a pre-aggregated daily zone table.

If a cleaned trip-level table is available, we can still use it here for metric design, but the notebook should stay scoped and memory-aware.

In [2]:
taxi_path = paths.processed / "taxi" / "clean"
taxi_files = sorted(taxi_path.rglob("*.parquet"))

print(f"Found {len(taxi_files)} cleaned taxi files")

taxi = pd.concat((pd.read_parquet(f) for f in taxi_files), ignore_index=True)

print(taxi.shape)
taxi.head()

Found 0 cleaned taxi files


ValueError: No objects to concatenate